# Fraud Detection Guided Notebook (CatBoost)

불균형(약 1.1%) 데이터셋에 대해 권장 전처리와 CatBoost 중심 학습을 적용합니다.

## 1) 환경 설정 및 라이브러리 임포트

In [ ]:
import json
import random
import warnings
from pathlib import Path

import joblib
import numpy as np
import optuna
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

USE_OPTUNA = True
N_SPLITS = 5
N_TRIALS = 35

DATA_DIR = Path(".")
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
SAMPLE_SUB_PATH = DATA_DIR / "sample_submission.csv"

print("seed:", SEED)
print("use_optuna:", USE_OPTUNA)

## 2) 데이터 로드 및 기본 검증

In [ ]:
train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)

print("train shape:", train_raw.shape)
print("test shape :", test_raw.shape)
print("target mean:", train_raw["fraud_bool"].mean())
print("class counts:")
print(train_raw["fraud_bool"].value_counts())
print("\nTop missing columns (train):")
print(train_raw.isna().mean().sort_values(ascending=False).head(12))

## 3) 권장 전처리 파이프라인 정의

- 무의미 변수 제거 (`id`, `device_fraud_history`)
- 센티넬(-1) 결측 플래그 생성
- 고위험 이진 플래그 생성
- 클립 + log1p 변환
- 희귀 범주 병합 + 원핫
- 구간화(Binning) + 상호작용 피처

In [ ]:
def merge_rare_categories(train_s: pd.Series, test_s: pd.Series, min_freq: float = 0.005):
    freq = train_s.value_counts(normalize=True)
    rare = set(freq[freq < min_freq].index)
    tr = train_s.where(~train_s.isin(rare), "Other")
    te = test_s.where(~test_s.isin(rare), "Other")
    return tr, te


def clip_log_transform(train_df: pd.DataFrame, test_df: pd.DataFrame, col: str, q_low=0.0, q_high=0.995):
    lo = float(train_df[col].quantile(q_low))
    hi = float(train_df[col].quantile(q_high))
    train_df[col] = np.log1p(train_df[col].clip(lo, hi))
    test_df[col] = np.log1p(test_df[col].clip(lo, hi))


def preprocess_guided(train_df: pd.DataFrame, test_df: pd.DataFrame):
    train = train_df.copy()
    test = test_df.copy()

    # remove uninformative and id-like columns
    drop_base = ["id", "device_fraud_history"]
    for c in drop_base:
        if c in train.columns:
            train = train.drop(columns=c)
        if c in test.columns:
            test = test.drop(columns=c)

    # sentinel(-1) handling with missing flags
    for c in ["prev_addr_months", "bank_months_count"]:
        if c in train.columns:
            med = train.loc[train[c] != -1, c].median()
            train[f"is_missing_{c}"] = (train[c] == -1).astype(np.int8)
            test[f"is_missing_{c}"] = (test[c] == -1).astype(np.int8)
            train[c] = train[c].where(train[c] != -1, med)
            test[c] = test[c].where(test[c] != -1, med)

    # key binary fraud flags
    if "name_email_sim" in train.columns:
        train["flag_name_email_low"] = (train["name_email_sim"] < 0.3).astype(np.int8)
        test["flag_name_email_low"] = (test["name_email_sim"] < 0.3).astype(np.int8)
        train["name_email_sim"] = train["name_email_sim"].clip(0, 1)
        test["name_email_sim"] = test["name_email_sim"].clip(0, 1)

    if "credit_risk_score" in train.columns:
        train["flag_credit_risk_lt0"] = (train["credit_risk_score"] < 0).astype(np.int8)
        test["flag_credit_risk_lt0"] = (test["credit_risk_score"] < 0).astype(np.int8)
        train["credit_risk_score"] = train["credit_risk_score"].clip(lower=0)
        test["credit_risk_score"] = test["credit_risk_score"].clip(lower=0)

    if "days_since_req" in train.columns:
        train["flag_days_since_req_lt7"] = (train["days_since_req"] < 7).astype(np.int8)
        test["flag_days_since_req_lt7"] = (test["days_since_req"] < 7).astype(np.int8)

    if "init_transfer_amt" in train.columns:
        threshold = float(train.loc[train["init_transfer_amt"] >= 0, "init_transfer_amt"].quantile(0.95))
        train["flag_init_transfer_high"] = (train["init_transfer_amt"] >= threshold).astype(np.int8)
        test["flag_init_transfer_high"] = (test["init_transfer_amt"] >= threshold).astype(np.int8)
        train["init_transfer_amt"] = train["init_transfer_amt"].clip(lower=0)
        test["init_transfer_amt"] = test["init_transfer_amt"].clip(lower=0)

    # yearly income U-shape flags
    if "yearly_income" in train.columns:
        y_lo = float(train["yearly_income"].quantile(0.1))
        y_hi = float(train["yearly_income"].quantile(0.9))
        train["flag_income_low"] = (train["yearly_income"] <= y_lo).astype(np.int8)
        train["flag_income_high"] = (train["yearly_income"] >= y_hi).astype(np.int8)
        test["flag_income_low"] = (test["yearly_income"] <= y_lo).astype(np.int8)
        test["flag_income_high"] = (test["yearly_income"] >= y_hi).astype(np.int8)

    # high risk category flags
    mapping_flags = [
        ("housing_status", "BA", "flag_housing_BA"),
        ("device_os", "windows", "flag_os_windows"),
        ("employment_status", "CC", "flag_employment_CC"),
        ("payment_type", "AC", "flag_payment_AC"),
        ("application_source", "TELEAPP", "flag_source_TELEAPP"),
    ]
    for col, val, out_col in mapping_flags:
        if col in train.columns:
            train[out_col] = (train[col].astype(str) == val).astype(np.int8)
            test[out_col] = (test[col].astype(str) == val).astype(np.int8)

    # skewed variable clip + log1p
    skew_cols = [
        "curr_addr_months", "session_length_min", "req_credit_limit", "branch_req_count_8w",
        "days_since_req", "dob_email_count_4w", "init_transfer_amt", "zip_req_count_4w"
    ]
    for c in skew_cols:
        if c in train.columns:
            clip_log_transform(train, test, c, q_low=0.0, q_high=0.995)

    # multimodal/binning candidates
    for c in ["req_rate_4w", "req_rate_24h"]:
        if c in train.columns:
            edges = np.unique(train[c].quantile([0.0, 0.33, 0.66, 1.0]).values)
            if len(edges) >= 3:
                train[f"{c}_bin"] = pd.cut(train[c], bins=edges, include_lowest=True, labels=False).astype("float")
                test[f"{c}_bin"] = pd.cut(test[c], bins=edges, include_lowest=True, labels=False).astype("float")
                train[f"{c}_bin"] = train[f"{c}_bin"].fillna(0).astype(np.int8)
                test[f"{c}_bin"] = test[f"{c}_bin"].fillna(0).astype(np.int8)

    # interaction features
    if "req_credit_limit" in train.columns and "credit_risk_score" in train.columns:
        train["feat_limit_over_credit"] = train["req_credit_limit"] / (train["credit_risk_score"] + 1e-3)
        test["feat_limit_over_credit"] = test["req_credit_limit"] / (test["credit_risk_score"] + 1e-3)

    if "credit_risk_score" in train.columns and "age_bucket" in train.columns:
        age_tr = pd.to_numeric(train["age_bucket"], errors="coerce").fillna(0)
        age_te = pd.to_numeric(test["age_bucket"], errors="coerce").fillna(0)
        train["feat_credit_x_age"] = train["credit_risk_score"] * age_tr
        test["feat_credit_x_age"] = test["credit_risk_score"] * age_te

    if "curr_addr_months" in train.columns and "name_email_sim" in train.columns:
        train["feat_curr_x_name_sim"] = train["curr_addr_months"] * train["name_email_sim"]
        test["feat_curr_x_name_sim"] = test["curr_addr_months"] * test["name_email_sim"]

    # rare category merge for key columns
    for c in ["housing_status", "device_os", "employment_status", "payment_type", "application_source"]:
        if c in train.columns:
            train[c], test[c] = merge_rare_categories(train[c].astype(str), test[c].astype(str), min_freq=0.005)

    # one-hot encode categories
    target_col = "fraud_bool"
    cat_cols = [c for c in train.columns if c != target_col and train[c].dtype == "object"]
    if cat_cols:
        train = pd.get_dummies(train, columns=cat_cols, dummy_na=False)
        test = pd.get_dummies(test, columns=cat_cols, dummy_na=False)

    y = train[target_col].astype(np.int8)
    X = train.drop(columns=[target_col], errors="ignore")
    X_test = test.copy()

    X, X_test = X.align(X_test, join="left", axis=1, fill_value=0)
    X = X.astype(np.float32)
    X_test = X_test.astype(np.float32)

    meta = {
        "seed": SEED,
        "n_features": int(X.shape[1]),
        "features": X.columns.tolist(),
        "dropped": drop_base,
        "cat_cols_encoded": cat_cols,
    }
    return X, y, X_test, meta


X, y, X_test, preprocess_meta = preprocess_guided(train_raw, test_raw)
print("X:", X.shape, "X_test:", X_test.shape)
print("fraud ratio:", y.mean())

## 4) 학습/검증 설정 및 평가 지표

In [ ]:
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)


def ap_metric(y_true, y_prob):
    return average_precision_score(y_true, y_prob)


if len(X) > 150000:
    rs = np.random.RandomState(SEED)
    idx = rs.choice(len(X), 150000, replace=False)
    X_tune = X.iloc[idx].reset_index(drop=True)
    y_tune = y.iloc[idx].reset_index(drop=True)
else:
    X_tune = X.copy()
    y_tune = y.copy()

print("tuning sample:", X_tune.shape)
print("class ratio:", y_tune.mean())

## 5) CatBoost Optuna 튜닝 (선택) + 6) 최종 5-Fold 학습

In [ ]:
def cat_objective(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 800, 2600),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "depth": trial.suggest_int("depth", 3, 8),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 30.0, log=True),
        "random_strength": trial.suggest_float("random_strength", 0.1, 5.0),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 5.0),
        "border_count": trial.suggest_int("border_count", 32, 255),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "bootstrap_type": "Bernoulli",
        "loss_function": "Logloss",
        "eval_metric": "PRAUC",
        "random_seed": SEED,
        "auto_class_weights": "Balanced",
        "verbose": False,
    }

    cv3 = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
    fold_scores = []
    for tr_idx, va_idx in cv3.split(X_tune, y_tune):
        X_tr, X_va = X_tune.iloc[tr_idx], X_tune.iloc[va_idx]
        y_tr, y_va = y_tune.iloc[tr_idx], y_tune.iloc[va_idx]
        model = CatBoostClassifier(**params)
        model.fit(X_tr, y_tr, eval_set=(X_va, y_va), verbose=False)
        pred = model.predict_proba(X_va)[:, 1]
        fold_scores.append(ap_metric(y_va, pred))
    return float(np.mean(fold_scores))


if USE_OPTUNA:
    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(cat_objective, n_trials=N_TRIALS, show_progress_bar=True)
    best_params = study.best_trial.params.copy()
    best_params.update({
        "bootstrap_type": "Bernoulli",
        "loss_function": "Logloss",
        "eval_metric": "PRAUC",
        "random_seed": SEED,
        "auto_class_weights": "Balanced",
        "early_stopping_rounds": 80,
        "verbose": False,
    })
else:
    study = None
    best_params = {
        "iterations": 2000,
        "learning_rate": 0.040331443383617174,
        "depth": 4,
        "l2_leaf_reg": 8.234548958371454,
        "random_strength": 0.514960710230099,
        "border_count": 87,
        "subsample": 0.8233119185389446,
        "bootstrap_type": "Bernoulli",
        "loss_function": "Logloss",
        "eval_metric": "PRAUC",
        "random_seed": SEED,
        "auto_class_weights": "Balanced",
        "early_stopping_rounds": 80,
        "verbose": False,
    }

print("best params:")
print(best_params)

oof = np.zeros(len(X), dtype=np.float64)
test_pred = np.zeros(len(X_test), dtype=np.float64)
fold_ap, fold_auc = [], []
models = []

for fold, (tr_idx, va_idx) in enumerate(cv.split(X, y), start=1):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

    model = CatBoostClassifier(**best_params)
    model.fit(X_tr, y_tr, eval_set=(X_va, y_va), verbose=False)

    va_pred = model.predict_proba(X_va)[:, 1]
    te_pred = model.predict_proba(X_test)[:, 1]

    oof[va_idx] = va_pred
    test_pred += te_pred / N_SPLITS
    models.append(model)

    ap = ap_metric(y_va, va_pred)
    auc = roc_auc_score(y_va, va_pred)
    fold_ap.append(ap)
    fold_auc.append(auc)
    print(f"fold {fold}: AP={ap:.6f}, ROC-AUC={auc:.6f}")

oof_ap = ap_metric(y, oof)
oof_auc = roc_auc_score(y, oof)
print(f"\nOOF AP: {oof_ap:.6f}")
print(f"OOF ROC-AUC: {oof_auc:.6f}")

## 7) 제출 파일/모델/리포트 저장

In [ ]:
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)
sub_col = "fraud_bool" if "fraud_bool" in sample_sub.columns else sample_sub.columns[-1]

submission = sample_sub.copy()
submission[sub_col] = test_pred
submission_path = DATA_DIR / "submission_fraud_catboost_guided.csv"
submission.to_csv(submission_path, index=False)

# full-data fit for deployment artifact
final_model = CatBoostClassifier(**best_params)
final_model.fit(X, y, verbose=False)
joblib.dump(final_model, DATA_DIR / "fraud_catboost_guided_model.joblib")

report = {
    "seed": SEED,
    "target_ratio": float(y.mean()),
    "n_features": int(X.shape[1]),
    "oof_ap": float(oof_ap),
    "oof_roc_auc": float(oof_auc),
    "fold_ap_mean": float(np.mean(fold_ap)),
    "fold_ap_std": float(np.std(fold_ap)),
    "fold_auc_mean": float(np.mean(fold_auc)),
    "fold_auc_std": float(np.std(fold_auc)),
    "best_params": best_params,
    "optuna_best_value": float(study.best_value) if study is not None else None,
    "files": {
        "submission": str(submission_path),
        "model": "fraud_catboost_guided_model.joblib",
    },
}

with open(DATA_DIR / "fraud_catboost_guided_report.json", "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

with open(DATA_DIR / "fraud_catboost_guided_preprocess_meta.json", "w", encoding="utf-8") as f:
    json.dump(preprocess_meta, f, ensure_ascii=False, indent=2)

print("saved:")
print(" -", submission_path)
print(" - fraud_catboost_guided_model.joblib")
print(" - fraud_catboost_guided_report.json")
print(" - fraud_catboost_guided_preprocess_meta.json")